In [1]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
from IPython.display import display


# ==========================================
# Sub-period analysis (display-only, no save)
# ==========================================
# Slices OOS daily returns into:
# 2009-2013, 2014-2019, 2020-2026
# for all assets/runs in Results_Daily and recomputes performance metrics.

MARKETS = ["QQQ", "NKY", "EUSTX"]
ANNUALIZATION = 252

SUB_PERIODS = [
    ("2009-2013", "2009-01-01", "2013-12-31"),
    ("2014-2019", "2014-01-01", "2019-12-31"),
    ("2020-2026", "2020-01-01", "2026-12-31"),
]


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "Results_Daily").exists() and (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root.")


def equity_to_returns(tab):
    tab = np.array(tab, dtype=np.float64)
    return (tab[1:] / tab[:-1]) - 1


def absolute_return(tab):
    ret = equity_to_returns(tab)
    if len(ret) == 0:
        return 0.0
    return (np.prod(1 + ret) - 1.0) * 100.0


def arc(tab, annualization=252):
    tab = np.array(tab, dtype=np.float64)
    ret = equity_to_returns(tab)
    length = len(tab)
    if length <= 1:
        return 0.0
    a_rtn = np.prod(1 + ret[:-1]) if len(ret) > 1 else 1.0
    if a_rtn <= 0:
        return 0.0
    return 100.0 * (math.pow(a_rtn, annualization / length) - 1.0)


def maximum_drawdown(tab):
    eqr = equity_to_returns(tab)
    if len(eqr) == 0:
        return 0.0
    cum_returns = np.cumprod(1 + eqr)
    cum_max = np.maximum.accumulate(cum_returns)
    drawdowns = (cum_max - cum_returns) / cum_max
    return np.max(drawdowns) * 100.0


def asd(tab, annualization=252):
    ret = equity_to_returns(tab)
    if len(ret) == 0:
        return 0.0
    return (np.sqrt(annualization) * np.std(ret)) * 100.0


def sgn(x):
    if x == 0:
        return 0
    return int(abs(x) / x)


def mld(tab, annualization=252):
    temp = np.array(tab, dtype=np.float64)
    sessions_per_year = annualization
    if len(temp) == 0:
        return 1.0
    i = np.argmax(np.maximum.accumulate(temp) - temp)
    if i == 0:
        return len(temp) / sessions_per_year
    j = np.argmax(temp[:i])
    mld_end = -1
    for k in range(i, len(temp)):
        if (temp[k - 1] < temp[j]) and (temp[j] < temp[k]):
            mld_end = k
            break
    if mld_end == -1:
        mld_end = len(temp)
    return abs(mld_end - j) / sessions_per_year


def ir1(tab, annualization=252):
    asd_v = asd(tab, annualization)
    arc_v = arc(tab, annualization)
    if asd_v == 0:
        return 0.0
    return max(arc_v / asd_v, 0.0)


def ir2(tab, annualization=252):
    asd_v = asd(tab, annualization)
    arc_v = arc(tab, annualization)
    mdd_v = maximum_drawdown(tab)
    denom = asd_v * mdd_v
    if denom == 0:
        return 0.0
    numer = (arc_v ** 2) * sgn(arc_v)
    return max(numer / denom, 0.0)


def ir3(tab, annualization=252):
    asd_v = asd(tab, annualization)
    arc_v = arc(tab, annualization)
    mdd_v = maximum_drawdown(tab)
    mld_v = mld(tab, annualization)
    denom = asd_v * mdd_v * mld_v
    if denom == 0:
        return 0.0
    return (arc_v ** 3) / denom


def compute_metrics_from_returns(returns: np.ndarray, annualization: int = 252) -> dict:
    r = np.asarray(returns, dtype=float)
    eq = np.concatenate(([1.0], np.cumprod(1.0 + r)))

    if len(r) > 1 and np.std(r) > 0:
        sharpe = (np.mean(r) / np.std(r)) * np.sqrt(annualization)
    else:
        sharpe = 0.0

    arc_v = arc(eq, annualization)
    mdd_v = maximum_drawdown(eq)

    return {
        "Absolute Return (%)": round(absolute_return(eq), 4),
        "ARC (%)": round(arc_v, 4),
        "ASD (%)": round(asd(eq, annualization), 4),
        "Max Drawdown (%)": round(mdd_v, 4),
        "MLD (years)": round(mld(eq, annualization), 4),
        "IR1": round(ir1(eq, annualization) * 100.0, 4),
        "IR2": round(ir2(eq, annualization) * 100.0, 4),
        "IR3": round(ir3(eq, annualization) * 100.0, 4),
        "Sharpe": round(sharpe, 4),
        "N Days": int(len(r)),
    }


repo_root = find_repo_root(Path.cwd().resolve())
results_root = repo_root / "Results_Daily"

# NKY custom benchmark (EWJ_NK): convert close prices to daily returns
ewj_benchmark_path = repo_root / "v1_Changes_Calculations" / "new_benchamrk_data" / "EWJ_NK.csv"
ewj_benchmark = pd.read_csv(ewj_benchmark_path)
ewj_benchmark["date"] = pd.to_datetime(ewj_benchmark["date"])
ewj_benchmark = ewj_benchmark.sort_values("date")
ewj_benchmark["EWJ_NK"] = ewj_benchmark["close"].pct_change().fillna(0.0)
ewj_benchmark = ewj_benchmark[["date", "EWJ_NK"]]

rows = []

for market in MARKETS:
    market_dir = results_root / market
    if not market_dir.exists():
        continue

    run_dirs = sorted([p for p in market_dir.iterdir() if p.is_dir()])

    # Market-level Markowitz benchmark series
    mkz_path = repo_root / "v1_Changes_Calculations" / "Benchmark_Markowitz" / f"markowitz_daily_returns_{market}.csv"
    mkz_col = f"{market}_Min Variance"
    mkz_df = None
    if mkz_path.exists():
        mkz_df = pd.read_csv(mkz_path)
        mkz_df["date"] = pd.to_datetime(mkz_df["date"])
        mkz_df = mkz_df[["date", mkz_col]].rename(columns={mkz_col: "Markowitz Min Variance"})

    for run_dir in run_dirs:
        rl_path = run_dir / "rl_daily_returns_oos.csv"
        base_path = run_dir / "daily_returns_oos.csv"
        if not rl_path.exists():
            continue

        df_rl = pd.read_csv(rl_path)
        if "date" not in df_rl.columns or "RL Agent" not in df_rl.columns:
            continue
        df_rl["date"] = pd.to_datetime(df_rl["date"])

        # Start from RL core frame
        df = df_rl[["date", "RL Agent"]].copy()

        # Add baseline benchmark returns (if available)
        if base_path.exists():
            df_base = pd.read_csv(base_path)
            df_base["date"] = pd.to_datetime(df_base["date"])
            keep = [
                "date",
                "QQQ Buy-and-Hold",
                "Momentum Top-20%",
                "Equal-Weight Monthly",
            ]
            keep = [c for c in keep if c in df_base.columns]
            if len(keep) > 1:
                rename_map = {
                    "QQQ Buy-and-Hold": "Buy & Hold",
                    "Momentum Top-20%": "Momentum Top-20",
                    "Equal-Weight Monthly": "Equal-Weight Monthly",
                }
                df = df.merge(df_base[keep].rename(columns=rename_map), on="date", how="left")

        # Override NKY buy-hold benchmark with EWJ_NK
        if market == "NKY":
            df = df.merge(ewj_benchmark, on="date", how="left")
            df["EWJ_NK"] = df["EWJ_NK"].ffill().bfill()
            df = df.drop(columns=["Buy & Hold"], errors="ignore")
            df = df.rename(columns={"EWJ_NK": "Buy & Hold"})

        # Add Markowitz benchmark
        if mkz_df is not None:
            df = df.merge(mkz_df, on="date", how="left")

        benchmark_cols = [
            c for c in ["Buy & Hold", "Momentum Top-20", "Equal-Weight Monthly", "Markowitz Min Variance"]
            if c in df.columns
        ]

        for period_name, start, end in SUB_PERIODS:
            mask = (df["date"] >= pd.Timestamp(start)) & (df["date"] <= pd.Timestamp(end))
            sub = df.loc[mask].copy()

            if sub.empty:
                continue

            # RL metrics
            rl_metrics = compute_metrics_from_returns(sub["RL Agent"].to_numpy(dtype=float), ANNUALIZATION)
            rows.append(
                {
                    "Market": market,
                    "Run": run_dir.name,
                    "Series": "RL Agent",
                    "Benchmark": "ALL",
                    "Period": period_name,
                    "Start": sub["date"].min().date(),
                    "End": sub["date"].max().date(),
                    **rl_metrics,
                }
            )

            # Benchmark metrics (all available benchmarks)
            for bcol in benchmark_cols:
                b = sub[bcol].astype(float).replace([np.inf, -np.inf], np.nan).dropna()
                if b.empty:
                    continue
                b_metrics = compute_metrics_from_returns(b.to_numpy(dtype=float), ANNUALIZATION)
                rows.append(
                    {
                        "Market": market,
                        "Run": run_dir.name,
                        "Series": bcol,
                        "Benchmark": bcol,
                        "Period": period_name,
                        "Start": sub["date"].min().date(),
                        "End": sub["date"].max().date(),
                        **b_metrics,
                    }
                )

subperiod_results = (
    pd.DataFrame(rows)
    .sort_values(["Market", "Run", "Series", "Period"])
    .reset_index(drop=True)
)

print("Sub-period analysis complete (no file saved)")
print(f"Repo: {repo_root}")
print(f"Rows: {len(subperiod_results)}")

print("\nFull sub-period metrics")
display(subperiod_results)

# Compact thesis-ready pivot for RL only
rl_only = subperiod_results.loc[subperiod_results["Series"] == "RL Agent"].copy()
absret_pivot = rl_only.pivot_table(index=["Market", "Run"], columns="Period", values="Absolute Return (%)")
arc_pivot = rl_only.pivot_table(index=["Market", "Run"], columns="Period", values="ARC (%)")
ir2_pivot = rl_only.pivot_table(index=["Market", "Run"], columns="Period", values="IR2")
sharpe_pivot = rl_only.pivot_table(index=["Market", "Run"], columns="Period", values="Sharpe")

print("\nRL Absolute Return (%) by sub-period")
display(absret_pivot)

print("\nRL ARC (%) by sub-period")
display(arc_pivot)

print("\nRL IR2 by sub-period")
display(ir2_pivot)

print("\nRL Sharpe by sub-period")
display(sharpe_pivot)

# Optional relative view: RL - benchmark (market-specific)
rl_key = rl_only.set_index(["Market", "Run", "Period"])
bench_all = subperiod_results.loc[subperiod_results["Series"] != "RL Agent"].copy()

rel_rows = []
for idx, rl_row in rl_key.iterrows():
    market, run, period = idx

    b_rows = bench_all.loc[
        (bench_all["Market"] == market)
        & (bench_all["Run"] == run)
        & (bench_all["Period"] == period)
    ]

    if b_rows.empty:
        continue

    for _, b0 in b_rows.iterrows():
        rel_rows.append(
            {
                "Market": market,
                "Run": run,
                "Period": period,
                "Benchmark": b0["Series"],
                "ARC Diff (RL-Benchmark)": rl_row["ARC (%)"] - b0["ARC (%)"],
                "IR2 Diff (RL-Benchmark)": rl_row["IR2"] - b0["IR2"],
                "IR3 Diff (RL-Benchmark)": rl_row["IR3"] - b0["IR3"],
                "Sharpe Diff (RL-Benchmark)": rl_row["Sharpe"] - b0["Sharpe"],
            }
        )

if rel_rows:
    rel = pd.DataFrame(rel_rows).sort_values(["Market", "Run", "Period", "Benchmark"]).reset_index(drop=True)
    print("\nRelative performance (RL - each benchmark) by sub-period")
    display(rel)

Sub-period analysis complete (no file saved)
Repo: /Users/kamilkashif/Documents/University/Masters Thesis/MS-Thesis-Deep-RL-KK
Rows: 135

Full sub-period metrics


,Market,Run,Series,Benchmark,Period,Start,End,Absolute Return (%),ARC (%),ASD (%),Max Drawdown (%),MLD (years),IR1,IR2,IR3,Sharpe,N Days
0,EUSTX,EUSTX_LSTM_DAILY_1,Buy & Hold,Buy & Hold,2009-2013,2009-03-31,2013-12-31,85.8558,13.6501,30.4200,37.8020,2.4881,44.8720,16.2030,88.8921,0.5750,1215
1,EUSTX,EUSTX_LSTM_DAILY_1,Buy & Hold,Buy & Hold,2014-2019,2014-01-02,2019-12-31,11.2221,1.6838,17.1262,30.8350,3.2976,9.8315,0.5369,0.2741,0.1887,1530
2,EUSTX,EUSTX_LSTM_DAILY_1,Buy & Hold,Buy & Hold,2020-2026,2020-01-02,2025-03-28,47.3724,7.6585,23.9063,38.9481,2.1746,32.0354,6.2992,22.1845,0.4256,1340
3,EUSTX,EUSTX_LSTM_DAILY_1,Equal-Weight Monthly,Equal-Weight Monthly,2009-2013,2009-03-31,2013-12-31,101.1661,15.4959,24.2838,34.7110,2.2619,63.8117,28.4872,195.1601,0.7184,1215
4,EUSTX,EUSTX_LSTM_DAILY_1,Equal-Weight Monthly,Equal-Weight Monthly,2014-2019,2014-01-02,2019-12-31,51.6620,7.1215,17.1233,25.1093,1.9722,41.5895,11.7956,42.5925,0.4866,1530
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130,QQQ,QQQ_TRANSFORMERS_DAILY_1,Momentum Top-20,Momentum Top-20,2014-2019,2014-01-02,2019-12-31,57.9400,7.8997,17.2319,27.7676,1.4325,45.8436,13.0422,71.9213,0.5309,1504
131,QQQ,QQQ_TRANSFORMERS_DAILY_1,Momentum Top-20,Momentum Top-20,2020-2026,2020-01-02,2025-04-01,34.5256,5.6687,25.9640,40.7443,2.2540,21.8328,3.0376,7.6394,0.3495,1314
132,QQQ,QQQ_TRANSFORMERS_DAILY_1,RL Agent,ALL,2009-2013,2009-04-03,2013-12-31,212.9948,27.0910,20.1559,17.3083,0.9127,134.4072,210.3754,6244.4373,1.2993,1191
133,QQQ,QQQ_TRANSFORMERS_DAILY_1,RL Agent,ALL,2014-2019,2014-01-02,2019-12-31,122.5754,14.2743,17.2607,23.5663,1.2143,82.6984,50.0911,588.8377,0.8635,1504



RL Absolute Return (%) by sub-period


Period                             2009-2013  2014-2019  2020-2026
Market Run                                                        
EUSTX  EUSTX_LSTM_DAILY_1            67.4813    53.4745    46.0820
       EUSTX_LSTM_DAILY_2            69.6078    52.1492    41.6005
       EUSTX_TRANSFORMERS_DAILY_1    71.7691    52.7633    46.6033
NKY    NKY_LSTM_DAILY_1              64.4293    62.9789    93.3571
       NKY_LSTM_DAILY_2              28.9504    63.5478    82.9287
       NKY_TRANSFORMERS_DAILY_1      40.3223    48.2826    84.4511
QQQ    QQQ_LSTM_DAILY_1             222.9149   122.4171    85.8556
       QQQ_LSTM_DAILY_2             175.9299   115.0209    71.8946
       QQQ_TRANSFORMERS_DAILY_1     212.9948   122.5754    87.6003


RL ARC (%) by sub-period


...

Period                             2009-2013  2014-2019  2020-2026
Market Run                                                        
EUSTX  EUSTX_LSTM_DAILY_1            21.3088    14.6658     9.1032
       EUSTX_LSTM_DAILY_2            30.0568    20.2084    10.0782
       EUSTX_TRANSFORMERS_DAILY_1    23.4753    13.9544     9.3842
NKY    NKY_LSTM_DAILY_1              15.9857    15.7499    30.3481
       NKY_LSTM_DAILY_2               3.9539    26.5324    32.5616
       NKY_TRANSFORMERS_DAILY_1       6.3216     8.2443    24.6235
QQQ    QQQ_LSTM_DAILY_1             215.4306    50.3017    17.9455
       QQQ_LSTM_DAILY_2             200.6518    61.8630    17.2254
       QQQ_TRANSFORMERS_DAILY_1     210.3754    50.0911    18.7099


RL Sharpe by sub-period


Period                             2009-2013  2014-2019  2020-2026
Market Run                                                        
EUSTX  EUSTX_LSTM_DAILY_1             0.6037     0.5163     0.4775
       EUSTX_LSTM_DAILY_2             0.6815     0.5696     0.4967
       EUSTX_TRANSFORMERS_DAILY_1     0.6257     0.5090     0.4813
NKY    NKY_LSTM_DAILY_1               0.5942     0.5192     0.7292
       NKY_LSTM_DAILY_2               0.3712     0.5864     0.7502
       NKY_TRANSFORMERS_DAILY_1       0.4327     0.4435     0.6597
QQQ    QQQ_LSTM_DAILY_1               1.3035     0.8654     0.5825
       QQQ_LSTM_DAILY_2               1.3221     0.9446     0.5640
       QQQ_TRANSFORMERS_DAILY_1       1.2993     0.8635     0.5914


Relative performance (RL - each benchmark) by sub-period


,Market,Run,Period,Benchmark,ARC Diff (RL-Benchmark),IR2 Diff (RL-Benchmark),IR3 Diff (RL-Benchmark),Sharpe Diff (RL-Benchmark)
0,EUSTX,EUSTX_LSTM_DAILY_1,2009-2013,Buy & Hold,-2.4211,5.1058,34.1638,0.0287
1,EUSTX,EUSTX_LSTM_DAILY_1,2009-2013,Equal-Weight Monthly,-4.2669,-7.1784,-72.1042,-0.1147
2,EUSTX,EUSTX_LSTM_DAILY_1,2009-2013,Markowitz Min Variance,-1.4229,-11.2652,-178.8479,-0.0877
3,EUSTX,EUSTX_LSTM_DAILY_1,2009-2013,Momentum Top-20,-5.9023,-18.6527,-307.1617,-0.1504
4,EUSTX,EUSTX_LSTM_DAILY_1,2014-2019,Buy & Hold,5.6332,14.1289,49.9896,0.3276
...,...,...,...,...,...,...,...,...
103,QQQ,QQQ_TRANSFORMERS_DAILY_1,2014-2019,Momentum Top-20,6.3746,37.0489,516.9164,0.3326
104,QQQ,QQQ_TRANSFORMERS_DAILY_1,2020-2026,Buy & Hold,-3.3899,-8.8128,-119.8096,-0.1250
105,QQQ,QQQ_TRANSFORMERS_DAILY_1,2020-2026,Equal-Weight Monthly,0.5965,1.4207,4.2285,0.0044
106,QQQ,QQQ_TRANSFORMERS_DAILY_1,2020-2026,Markowitz Min Variance,0.6041,-1.7665,-235.8470,-0.0259


In [5]:
subperiod_results[subperiod_results['Market']=='EUSTX']

,Market,Run,Series,Benchmark,Period,Start,End,Absolute Return (%),ARC (%),ASD (%),Max Drawdown (%),MLD (years),IR1,IR2,IR3,Sharpe,N Days
0,EUSTX,EUSTX_LSTM_DAILY_1,Buy & Hold,Buy & Hold,2009-2013,2009-03-31,2013-12-31,85.8558,13.6501,30.4200,37.8020,2.4881,44.8720,16.2030,88.8921,0.5750,1215
1,EUSTX,EUSTX_LSTM_DAILY_1,Buy & Hold,Buy & Hold,2014-2019,2014-01-02,2019-12-31,11.2221,1.6838,17.1262,30.8350,3.2976,9.8315,0.5369,0.2741,0.1887,1530
2,EUSTX,EUSTX_LSTM_DAILY_1,Buy & Hold,Buy & Hold,2020-2026,2020-01-02,2025-03-28,47.3724,7.6585,23.9063,38.9481,2.1746,32.0354,6.2992,22.1845,0.4256,1340
3,EUSTX,EUSTX_LSTM_DAILY_1,Equal-Weight Monthly,Equal-Weight Monthly,2009-2013,2009-03-31,2013-12-31,101.1661,15.4959,24.2838,34.7110,2.2619,63.8117,28.4872,195.1601,0.7184,1215
4,EUSTX,EUSTX_LSTM_DAILY_1,Equal-Weight Monthly,Equal-Weight Monthly,2014-2019,2014-01-02,2019-12-31,51.6620,7.1215,17.1233,25.1093,1.9722,41.5895,11.7956,42.5925,0.4866,1530
5,EUSTX,EUSTX_LSTM_DAILY_1,Equal-Weight Monthly,Equal-Weight Monthly,2020-2026,2020-01-02,2025-03-28,45.3575,7.4280,20.6840,39.3977,1.1627,35.9118,6.7708,43.2558,0.4443,1340
6,EUSTX,EUSTX_LSTM_DAILY_1,Markowitz Min Variance,Markowitz Min Variance,2009-2013,2009-03-31,2013-12-31,77.7057,12.6519,20.2393,24.2798,1.3651,62.5116,32.5740,301.9038,0.6914,1213
7,EUSTX,EUSTX_LSTM_DAILY_1,Markowitz Min Variance,Markowitz Min Variance,2014-2019,2014-01-02,2019-12-31,77.2834,9.9189,16.4760,23.5937,2.0833,60.2025,25.3095,120.5010,0.6569,1525
8,EUSTX,EUSTX_LSTM_DAILY_1,Markowitz Min Variance,Markowitz Min Variance,2020-2026,2020-01-02,2025-03-28,39.0833,6.6013,18.5851,36.2361,0.7262,35.5195,6.4708,58.8219,0.4286,1335
9,EUSTX,EUSTX_LSTM_DAILY_1,Momentum Top-20,Momentum Top-20,2009-2013,2009-03-31,2013-12-31,114.9814,17.1313,25.3087,29.0182,1.5913,67.6894,39.9615,430.2176,0.7541,1215
